In [1]:
import os

#!git clone https://github.com/recsyspolimi/RecSys_Course_AT_PoliMi

!pwd
os.chdir("../RecSys_Course_AT_PoliMi")
!pwd

#!python run_compile_all_cython.py

/Users/filippo/Documents/PoliMI/Recommender Systems/Challenge/hybrids_testing
/Users/filippo/Documents/PoliMI/Recommender Systems/Challenge/RecSys_Course_AT_PoliMi


In [2]:
import os
import time 
import numpy as np
import pandas as pd
import scipy.sparse as sp
import scipy.sparse as sps
import matplotlib.pyplot as pyplot
%matplotlib inline

from sklearn.model_selection import KFold
from Data_manager.split_functions.split_train_validation_random_holdout import split_train_in_two_percentage_global_sample
from skopt.space import Real, Integer, Categorical
from scipy.sparse import csr_matrix
from Evaluation.Evaluator import EvaluatorHoldout
from HyperparameterTuning.SearchBayesianSkopt import SearchBayesianSkopt
"""from Recommenders.SLIM.SLIMElasticNetRecommender import SLIMElasticNetRecommender
from Recommenders.EASE_R.EASE_R_Recommender import EASE_R_Recommender"""
#from Recommenders.GraphBased.RP3betaRecommender import RP3betaRecommender
from Recommenders.KNN.ItemKNNCBFRecommender import ItemKNNCBFRecommender
from Recommenders.KNN.ItemKNNCustomSimilarityRecommender import ItemKNNCustomSimilarityRecommender

from Recommenders.MatrixFactorization.IALSRecommender import FeatureCombinedImplicitALSRecommender

#from Recommenders.MatrixFactorization.IALSRecommender import IALSRecommender
from Recommenders.Neural.MultVAE_PyTorch_Recommender import MultVAERecommender_PyTorch

from Recommenders.hybrid.m2_model import IntegratedHierarchicalHybridRecommender
from Recommenders.hybrid.LinearHybridRecommender import NormalizedLinearCoupleHybridRecommender


Tensorflow is not available


In [ ]:
import numpy as np
import torch
import math
import torch.nn.functional as f
from Recommenders.Neural.MultVAE_PyTorch_Recommender import MultVAERecommender_PyTorch


"""

THIS IS A PATCH CAUSE OF SOME COMPATIBILITY ERRORS ON MULTVAE

"""

# 1. Fix per il training (_run_epoch)
def patched_run_epoch(self, num_epoch):
    num_batches_per_epoch = math.ceil(len(self.warm_user_ids) / self.batch_size)
    self._model.train()
    epoch_loss = 0

    for _ in range(num_batches_per_epoch):
        self._optimizer.zero_grad()

        # FIX: Usiamo un array numpy per indicizzare la matrice scipy
        u_idx = np.random.choice(self.warm_user_ids, size=self.batch_size)
        user_batch_tensor = self.URM_train[u_idx]

        user_batch_tensor = torch.sparse_csr_tensor(user_batch_tensor.indptr,
                                                    user_batch_tensor.indices,
                                                    user_batch_tensor.data,
                                                    size=user_batch_tensor.shape, 
                                                    dtype=torch.float32, 
                                                    device=self.device).to_dense()

        logits, KL, mu_q, std_q, epsilon, sampled_z = self._model.forward(user_batch_tensor)
        log_softmax_var = f.log_softmax(logits, dim=1)
        neg_ll = - torch.mean(torch.sum(log_softmax_var * user_batch_tensor, dim=1))
        l2_reg = self._model.get_l2_reg()
        anneal = min(self.anneal_cap, 1. * self.update_count / self.total_anneal_steps) if self.total_anneal_steps > 0 else self.anneal_cap

        loss = neg_ll + anneal * KL + l2_reg * self.l2_reg
        self.update_count += 1
        loss.backward()
        epoch_loss += loss.item()
        self._optimizer.step()

    self._print("Loss {:.2E}".format(epoch_loss))
    self._model.eval()

# 2. Fix per la predizione (_compute_item_score)
def patched_compute_item_score(self, user_id_array, items_to_compute = None):
    # FIX: Usiamo user_id_array direttamente (è già numpy)
    user_batch_tensor = self.URM_train[user_id_array]
    user_batch_tensor = torch.sparse_csr_tensor(user_batch_tensor.indptr,
                                                user_batch_tensor.indices,
                                                user_batch_tensor.data,
                                                size=user_batch_tensor.shape, 
                                                dtype=torch.float32,
                                                device=self.device).to_dense()

    with torch.no_grad():
        self._model.eval()
        logits, _, _, _, _, _ = self._model.forward(user_batch_tensor)

    item_scores_to_compute = logits.cpu().detach().numpy()
    if items_to_compute is not None:
        item_scores = - np.ones((len(user_id_array), self.n_items)) * np.inf
        item_scores[:, items_to_compute] = item_scores_to_compute[:, items_to_compute]
    else:
        item_scores = item_scores_to_compute
    return item_scores

# Sovrascriviamo i metodi nella classe base
MultVAERecommender_PyTorch._run_epoch = patched_run_epoch
MultVAERecommender_PyTorch._compute_item_score = patched_compute_item_score

In [4]:
df_train = pd.read_csv("data_train.csv")
df_test_user = pd.read_csv("data_target_users_test.csv")

# Training knn costum similarity

In [5]:
"""SLIM_params = {
    'topK': 625,
    'l1_ratio': 0.09517221375634205,
    'alpha': 0.00228698730766055
}

ease_params = {
    'topK': 1431,
    'l2_norm': 426.57622242296605
}"""

"""rp3_params = {
    'alpha': 0.7733352330682174,
    'beta': 0.4139018623121251,
    'topK': 35
}"""

IALS_params = {
    'iterations': 135,
    'factors': 87,
    'alpha': 7.762338288061237,
    'regularization': 0.004799745261257595
}

"""Trial 10 finished with value: 0.22947985424428655 and parameters: 
{'learning_rate': 2.2956778160991473e-05, 'l2_reg': 4.747958042741239e-05, 
'encoding_size': 597, 'next_layer_size_multiplier': 2.9916958009201124, 
'dropout': 0.4118674496820792, 'anneal_cap': 0.5969296800364489}."""

MultVAE_params = {
    'learning_rate': 2.2956778160991473e-05,
    'l2_reg': 4.747958042741239e-05, 
    'encoding_size': 597,
    'next_layer_size_multiplier': 2.9916958009201124,
    'dropout': 0.4118674496820792, 
    'anneal_cap': 0.5969296800364489
}

In [6]:
from scipy.sparse import coo_matrix

#valore 1 per ogni coppia (row, col)
data = [1] * len(df_train)
df_train["row"] = df_train["row"].astype(int)
df_train["col"] = df_train["col"].astype(int)

# matrice COO
URM_all = sp.csr_matrix((data, (df_train["row"], df_train["col"])))


In [7]:
def split_train_in_five_percentage_global_sample(URM_all, train_percentages):
    """
    The function splits an URM in five matrices based on provided percentages.
    :param URM_all: The full URM matrix
    :param train_percentages: A list of percentages (must sum to 1.0)
    :return: A list of 5 sparse matrices
    """

    import numpy as np
    from scipy.sparse import coo_matrix
    from Data_manager.IncrementalSparseMatrix import IncrementalSparseMatrix

    assert len(train_percentages) == 5, "You must provide exactly 5 percentages."
    assert abs(sum(train_percentages) - 1.0) < 1e-6, "Percentages must sum to 1.0."

    num_users, num_items = URM_all.shape

    # Builders for each of the 5 matrices
    builders = [
        IncrementalSparseMatrix(n_rows=num_users, n_cols=num_items, auto_create_col_mapper=False, auto_create_row_mapper=False)
        for _ in range(5)
    ]

    URM_all_coo = coo_matrix(URM_all)

    # Shuffle indices
    indices_for_sampling = np.arange(URM_all.nnz, dtype=np.int32)
    np.random.shuffle(indices_for_sampling)

    # Calculate the number of interactions for each split
    split_sizes = [int(URM_all.nnz * percentage) for percentage in train_percentages]
    cumulative_sizes = np.cumsum(split_sizes)

    # Divide the indices into 5 groups
    indices_splits = [
        indices_for_sampling[cumulative_sizes[i - 1]:cumulative_sizes[i]] if i > 0 else indices_for_sampling[:cumulative_sizes[i]]
        for i in range(5)
    ]

    # Populate the builders
    for i, builder in enumerate(builders):
        builder.add_data_lists(
            URM_all_coo.row[indices_splits[i]],
            URM_all_coo.col[indices_splits[i]],
            URM_all_coo.data[indices_splits[i]],
        )

    # Convert to sparse matrices
    sparse_matrices = [builder.get_SparseMatrix() for builder in builders]

    # Ensure all outputs are in csr_matrix format
    sparse_matrices = [sp.csr_matrix(matrix) for matrix in sparse_matrices]

    return sparse_matrices

In [8]:
train_percentages = [0.2, 0.2, 0.2, 0.2, 0.2]  # Cinque parti uguali

URM_parts = split_train_in_five_percentage_global_sample(URM_all, train_percentages)
URM_parts

[<Compressed Sparse Row sparse matrix of dtype 'float64'
 	with 608611 stored elements and shape (27095, 6969)>,
 <Compressed Sparse Row sparse matrix of dtype 'float64'
 	with 608611 stored elements and shape (27095, 6969)>,
 <Compressed Sparse Row sparse matrix of dtype 'float64'
 	with 608611 stored elements and shape (27095, 6969)>,
 <Compressed Sparse Row sparse matrix of dtype 'float64'
 	with 608611 stored elements and shape (27095, 6969)>,
 <Compressed Sparse Row sparse matrix of dtype 'float64'
 	with 608611 stored elements and shape (27095, 6969)>]

In [9]:
prefitted_folds = []
from Recommenders.Neural.MultVAE_PyTorch_Recommender import MultVAERecommender_PyTorch_OptimizerMask


for i in range(5):
    print(f"Fitting fold {i+1}/5...")
    
    # Creazione URM Train (Combined) e Test per questo fold
    URM_train = sum(URM_parts[j] for j in range(len(URM_parts)) if j != i)
    URM_test = URM_parts[i]
    
    # 1. Fit MultVAE
    recommender_multvae = MultVAERecommender_PyTorch_OptimizerMask(URM_train)
    recommender_multvae.fit(**MultVAE_params)
    
    # 2. Fit IALS
    als_recommender = FeatureCombinedImplicitALSRecommender(URM_train)
    print(f"Fitting ALS for fold {i+1}")
    als_recommender.fit(**IALS_params)
    
    evaluator_test = EvaluatorHoldout(URM_test, cutoff_list=[20])
    
    fold_data = {
        "URM_train": URM_train,
        "multvae": recommender_multvae,
        "ials": als_recommender,
        "evaluator": evaluator_test
    }
    
    prefitted_folds.append(fold_data)

print("Pre-training completato.")

Fitting fold 1/5...


/var/folders/wp/jydg89697jzcwnllv2_yz6d40000gn/T/ipykernel_8474/3626793068.py:27: UserWarning: Sparse CSR tensor support is in beta state. If you miss a functionality in the sparse tensor support, please submit a feature request to https://github.com/pytorch/pytorch/issues. (Triggered internally at /Users/runner/work/pytorch/pytorch/pytorch/aten/src/ATen/SparseCsrTensorImpl.cpp:55.)
  user_batch_tensor = torch.sparse_csr_tensor(user_batch_tensor.indptr,


MultVAERecommender_PyTorch: Epoch 1 of 10. Elapsed time 9.38 sec
MultVAERecommender_PyTorch: Epoch 2 of 10. Elapsed time 18.19 sec
MultVAERecommender_PyTorch: Epoch 3 of 10. Elapsed time 27.85 sec
MultVAERecommender_PyTorch: Epoch 4 of 10. Elapsed time 36.61 sec
MultVAERecommender_PyTorch: Epoch 5 of 10. Elapsed time 45.99 sec
MultVAERecommender_PyTorch: Epoch 6 of 10. Elapsed time 54.79 sec
MultVAERecommender_PyTorch: Epoch 7 of 10. Elapsed time 1.06 min
MultVAERecommender_PyTorch: Epoch 8 of 10. Elapsed time 1.21 min
MultVAERecommender_PyTorch: Epoch 9 of 10. Elapsed time 1.36 min
MultVAERecommender_PyTorch: Epoch 10 of 10. Elapsed time 1.51 min
MultVAERecommender_PyTorch: Terminating at epoch 10. Elapsed time 1.51 min
Fitting ALS for fold 1
EvaluatorHoldout: Ignoring 41 ( 0.2%) Users that have less than 1 test interactions
Fitting fold 2/5...
MultVAERecommender_PyTorch: Epoch 1 of 10. Elapsed time 9.31 sec
MultVAERecommender_PyTorch: Epoch 2 of 10. Elapsed time 18.47 sec
MultVAEReco

In [16]:
import time 

class SaveResults(object):
    
    def __init__(self):
        self.results_df = pd.DataFrame(columns=["result", "train_time (min)"])
    
    def __call__(self, optuna_study, optuna_trial):
        hyperparam_dict = optuna_trial.params.copy()
        hyperparam_dict["result"] = optuna_trial.values[0]
        
        # Retrieve the optimal number of epochs and training time from the "user attributes" of the trial
        #hyperparam_dict["epochs"] = optuna_trial.user_attrs["epochs"]
        hyperparam_dict["train_time (min)"] = optuna_trial.user_attrs["train_time (min)"]
        
        self.results_df.loc[len(self.results_df)] = hyperparam_dict
        
        
def objective_function(optuna_trial):

    start_time = time.time()
    scores = []
    for fold_data in prefitted_folds:
        
        # Recuperiamo i modelli e i dati dal dizionario
        URM_train = fold_data["URM_train"]
        recommender_multvae = fold_data["multvae"]
        als_recommender = fold_data["ials"]
        evaluator_test = fold_data["evaluator"]
        
        recommender = NormalizedLinearCoupleHybridRecommender(
            URM_train, 
            [recommender_multvae, als_recommender] 
)
        
        alpha=optuna_trial.suggest_float("alpha", 0, 0.0001)    
        recommender.fit(alpha)
        
        result, _ = evaluator_test.evaluateRecommender(recommender)
        #print("prova = ", result["MAP"].values[0])
        #print(i)
        #print(result["RECALL"])
        scores.append(result["RECALL"].values[0])
        #if result["MAP"].values[0] < 0.051:
        #    break
        
    # Add the number of epochs selected by earlystopping as a "user attribute" of the optuna trial
    #epochs = recommender_instance.get_early_stopping_final_epochs_dict()["epochs"]
    #optuna_trial.set_user_attr("epochs", epochs) 
    optuna_trial.set_user_attr("train_time (min)", (time.time() - start_time)/60) 
    print(scores)
    return sum(scores) / len(scores)


In [17]:
import optuna
import importlib

optuna_study = optuna.create_study(direction="maximize")
        
save_results = SaveResults()
        
optuna_study.optimize(objective_function,
                      callbacks=[save_results],
                      n_trials = 5)

[I 2025-12-30 16:15:56,913] A new study created in memory with name: no-name-79a0936f-6773-4b15-976e-a549d11ebfa0


EvaluatorHoldout: Processed 27054 (100.0%) in 7.87 sec. Users per second: 3437
EvaluatorHoldout: Processed 27068 (100.0%) in 7.49 sec. Users per second: 3612
EvaluatorHoldout: Processed 27059 (100.0%) in 7.49 sec. Users per second: 3614
EvaluatorHoldout: Processed 27052 (100.0%) in 7.49 sec. Users per second: 3612
EvaluatorHoldout: Processed 27061 (100.0%) in 7.50 sec. Users per second: 3609


[I 2025-12-30 16:16:34,901] Trial 0 finished with value: 0.24623287985553413 and parameters: {'alpha': 3.6042802725902104e-06}. Best is trial 0 with value: 0.24623287985553413.


[0.24619996562125748, 0.24638629211615268, 0.24601655044057072, 0.24638424121886987, 0.24617734988082]
EvaluatorHoldout: Processed 27054 (100.0%) in 7.46 sec. Users per second: 3626
EvaluatorHoldout: Processed 27068 (100.0%) in 7.48 sec. Users per second: 3621
EvaluatorHoldout: Processed 27059 (100.0%) in 7.50 sec. Users per second: 3607
EvaluatorHoldout: Processed 27052 (100.0%) in 7.46 sec. Users per second: 3627
EvaluatorHoldout: Processed 27061 (100.0%) in 7.48 sec. Users per second: 3619


[I 2025-12-30 16:17:12,347] Trial 1 finished with value: 0.24623299017279004 and parameters: {'alpha': 2.8342200266212858e-05}. Best is trial 1 with value: 0.24623299017279004.


...

[0.24619996562125748, 0.24638629211615268, 0.24601710202685012, 0.24638424121886987, 0.24617734988082]
EvaluatorHoldout: Processed 27054 (100.0%) in 7.66 sec. Users per second: 3531
EvaluatorHoldout: Processed 27068 (100.0%) in 7.67 sec. Users per second: 3528
EvaluatorHoldout: Processed 27059 (100.0%) in 7.67 sec. Users per second: 3529
EvaluatorHoldout: Processed 27052 (100.0%) in 7.65 sec. Users per second: 3534
EvaluatorHoldout: Processed 27061 (100.0%) in 7.72 sec. Users per second: 3505


[I 2025-12-30 16:18:28,445] Trial 3 finished with value: 0.24623367976043875 and parameters: {'alpha': 7.388821880401616e-05}. Best is trial 3 with value: 0.24623367976043875.


[0.24620508605219296, 0.2463879641503144, 0.24601710202685012, 0.2463860895110479, 0.24617215706178827]
EvaluatorHoldout: Processed 27054 (100.0%) in 7.76 sec. Users per second: 3485
EvaluatorHoldout: Processed 27068 (100.0%) in 7.61 sec. Users per second: 3556
EvaluatorHoldout: Processed 27059 (100.0%) in 7.77 sec. Users per second: 3483
EvaluatorHoldout: Processed 27052 (100.0%) in 7.64 sec. Users per second: 3543
EvaluatorHoldout: Processed 27061 (100.0%) in 7.55 sec. Users per second: 3583


[I 2025-12-30 16:19:06,894] Trial 4 finished with value: 0.2462328049235254 and parameters: {'alpha': 8.985805592842786e-05}. Best is trial 3 with value: 0.24623367976043875.


[0.24620467986416203, 0.2463879641503144, 0.24601402233679015, 0.2463860895110479, 0.2461712687553126]


# Da qui inizia il training su URM_all

In [18]:
return

SyntaxError: 'return' outside function (3438313781.py, line 1)

In [ ]:
best_alpha_test = 0.7074505665346519
best_beta_test = 0.9199376036548086
#Trial 8 finished with value: 0.2909141858548001 and parameters: {'alpha': 0.7074505665346519, 'beta': 0.9199376036548086}

In [ ]:
recommender_rp3_f = RP3betaRecommender(URM_all)
recommender_rp3_f.fit(**rp3_params)

In [ ]:
recommender_SLIM_f = SLIMElasticNetRecommender(URM_all)
recommender_SLIM_f.fit(**SLIM_params)

In [ ]:
# Train the final model 
new_similarity = (1 - best_alpha_test) * csr_matrix(recommender_rp3_f.W_sparse) + best_alpha_test * recommender_SLIM_f.W_sparse 

hybridrecommender_object_f = ItemKNNCustomSimilarityRecommender(URM_all)
hybridrecommender_object_f.fit(new_similarity)

In [ ]:
als_recommender_f = FeatureCombinedImplicitALSRecommender(URM_all)
als_recommender_f.fit(**IALS_params)

In [ ]:
recommenders = [hybridrecommender_object_f, als_recommender_f]

linear_comb_rec_f = GeneralizedLinearCoupleHybridRecommender(URM_all, recommenders)
linear_comb_rec_f.fit(best_beta_test)

In [ ]:
import time

start_time = time.time()
results = []
for user_id in df_test_user["user_id"].tolist():
        recommendations = linear_comb_rec_f.recommend(user_id, cutoff=20)       
       
        results.append((user_id, ' '.join(map(str, recommendations))))

df_recommendations = pd.DataFrame(results, columns=["user_id", "item_list"])
print(df_recommendations)
df_recommendations.to_csv("recommendations_ok.csv", index=False)

end_time = time.time()